# SwinIR SEM 微调 — 正式训练

GPU: 2×T4 | Checkpoint: 10k iter | 目标: 70k iter

**策略**：
1. 先跑 200 iter 试运行（验证训练能启动、ETA 合理）
2. 确认无误后，跑完整训练
3. 训练完成后下载 output 中的模型和日志

## 0. 环境准备（与 check.ipynb 相同）

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}, GPUs: {torch.cuda.device_count()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        mem = getattr(p, 'total_memory', getattr(p, 'total_mem', 0))
        print(f'  GPU {i}: {p.name} — {mem/1e9:.1f} GB')

In [ ]:
import os, shutil
WORK_DIR = '/kaggle/working'
os.chdir(WORK_DIR)

# 克隆仓库
if not os.path.exists('BasicSR'):
    !git clone https://github.com/Log-Dog012/BasicSR.git
    !cd BasicSR && git checkout cuda-sem-finetune

# 安装依赖
os.chdir('BasicSR')
!pip install -r requirements.txt -q
!pip install -e . -q
!pip install lpips timm -q
print('✅ 环境就绪')

In [ ]:
# 复制模型文件
MODEL_INPUT = '/kaggle/input/models/logdog012/swinir-finetune/pytorch/default/1'
swinir_zoo = os.path.join(WORK_DIR, 'BasicSR', 'SwinIR', 'model_zoo')
exp_dir = os.path.join(WORK_DIR, 'BasicSR', 'experiments', 'finetune_SwinIR_SRx4_SEM')

# 预训练权重
os.makedirs(swinir_zoo, exist_ok=True)
for root, dirs, files in os.walk(MODEL_INPUT):
    for f in files:
        if 'classicalSR' in f and f.endswith('.pth'):
            shutil.copy2(os.path.join(root, f), os.path.join(swinir_zoo, f))
            print(f'✅ 预训练: {f}')
            break

# Checkpoint
for name in ['net_g_10000.pth', '10000.state']:
    for root, dirs, files in os.walk(MODEL_INPUT):
        if name in files:
            dst_dir = os.path.join(exp_dir, 'models' if 'pth' in name else 'training_states')
            os.makedirs(dst_dir, exist_ok=True)
            shutil.copy2(os.path.join(root, name), os.path.join(dst_dir, name))
            print(f'✅ {name}')
            break
print('✅ 模型文件就绪')

## 1. 试运行（200 iter，确认训练正常）

In [ ]:
import yaml

cfg_path = 'options/train/SwinIR/finetune_SwinIR_SRx4_SEM.yml'
with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

# 临时改为试运行配置
cfg['train']['total_iter'] = 200      # 只跑 200 iter
cfg['logger']['print_freq'] = 50       # 每 50 iter 打印
cfg['logger']['save_checkpoint_freq'] = 200
cfg['val']['val_freq'] = 200
cfg['auto_resume'] = True

# 确保 checkpoint 在正确位置
print(f'模型权重: {cfg["path"]["pretrain_network_g"]}')
print(f'训练 HR: {cfg["datasets"]["train"]["dataroot_gt"]}')
print(f'num_gpu={cfg["num_gpu"]}, batch={cfg["datasets"]["train"]["batch_size_per_gpu"]}')
print(f'auto_resume={cfg["auto_resume"]}')

# 保存试运行配置
test_cfg_path = 'options/train/SwinIR/finetune_test_run.yml'
with open(test_cfg_path, 'w', encoding='utf-8') as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)
print(f'\n试运行配置已保存: {test_cfg_path}')

In [ ]:
# 试运行 200 iter
print('开始试运行 (200 iter)...')
print('='*50)
!python -m basicsr.train -opt options/train/SwinIR/finetune_test_run.yml

In [ ]:
# 检查试运行结果
log_dir = os.path.join(exp_dir)
logs = sorted([f for f in os.listdir(log_dir) if f.endswith('.log')])
if logs:
    latest_log = os.path.join(log_dir, logs[-1])
    with open(latest_log, 'r') as f:
        lines = f.readlines()
    # 打印最后 10 行
    print(f'日志: {logs[-1]}')
    print('='*60)
    for line in lines[-10:]:
        print(line.rstrip())
    
    # 检查是否有报错
    errors = [l for l in lines if 'Error' in l or 'Traceback' in l]
    if errors:
        print(f'\n❌ 发现 {len(errors)} 个错误！')
    else:
        # 提取 ETA
        eta_lines = [l for l in lines if 'eta' in l]
        if eta_lines:
            print(f'\n✅ 试运行成功！最后的 ETA: {eta_lines[-1].strip()}')
else:
    print('❌ 未找到日志文件')

## 2. 正式训练

试运行确认无误后，运行此 cell。训练 70k iter，预计 10-12 小时（2×T4）。

**注意**：Kaggle Notebook 有 12 小时限制。如果超时，`auto_resume: true` 会在下次运行时自动恢复。

In [ ]:
# 恢复正式配置
with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

print('正式训练配置:')
print(f'  total_iter: {cfg["train"]["total_iter"]}')
print(f'  lr: {cfg["train"]["optim_g"]["lr"]}')
print(f'  milestones: {cfg["train"]["scheduler"]["milestones"]}')
print(f'  num_gpu: {cfg["num_gpu"]}, batch: {cfg["datasets"]["train"]["batch_size_per_gpu"]}')
print(f'  auto_resume: {cfg["auto_resume"]}')

In [ ]:
# 正式训练
print('开始正式训练 (70k iter)...')
print('='*50)
!python -m basicsr.train -opt options/train/SwinIR/finetune_SwinIR_SRx4_SEM.yml

## 3. 结果分析

In [ ]:
# 提取所有验证结果
logs = sorted([f for f in os.listdir(exp_dir) if f.endswith('.log')])
if logs:
    latest_log = os.path.join(exp_dir, logs[-1])
    with open(latest_log, 'r') as f:
        content = f.read()
    
    # 提取验证 PSNR
    import re
    val_matches = re.findall(r'psnr:\s+([\d.]+)\s+Best:\s+([\d.]+)\s+@\s+(\d+)', content)
    
    if val_matches:
        print('验证结果:')
        print(f'{"Iter":<10} {"PSNR":<10} {"Best PSNR":<12}')
        print('-' * 35)
        best_psnr = 0
        for psnr_val, best_val, iter_val in val_matches:
            marker = ' ←' if float(psnr_val) == float(best_val) else ''
            print(f'{iter_val:<10} {psnr_val:<10} {best_val:<12}{marker}')
    
    # 最后几行
    lines = content.strip().split('\n')
    print(f'\n日志最后 5 行:')
    for line in lines[-5:]:
        print(line)
else:
    print('未找到日志')

## 4. 保存结果到 output

训练完成后，output 目录中的文件会被保存为 Kaggle Dataset，可以下载。

In [ ]:
# 将训练结果复制到 output 目录（Kaggle 会自动保存 output/）
output_dir = '/kaggle/working/output'
os.makedirs(output_dir, exist_ok=True)

# 复制最新模型和训练状态
models_dst = os.path.join(output_dir, 'models')
states_dst = os.path.join(output_dir, 'training_states')
shutil.copytree(models_dir, models_dst, dirs_exist_ok=True)
shutil.copytree(states_dir, states_dst, dirs_exist_ok=True)
print(f'✅ 模型已复制到 output/models/')
print(f'✅ 训练状态已复制到 output/training_states/')

# 复制日志
logs_dst = os.path.join(output_dir, 'logs')
os.makedirs(logs_dst, exist_ok=True)
for f in os.listdir(exp_dir):
    if f.endswith('.log'):
        shutil.copy2(os.path.join(exp_dir, f), os.path.join(logs_dst, f))
print(f'✅ 日志已复制到 output/logs/')

# 列出 output 内容
print(f'\noutput 目录内容:')
for root, dirs, files in os.walk(output_dir):
    level = root.replace(output_dir, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = '  ' * (level + 1)
    for file in sorted(files)[:10]:
        size = os.path.getsize(os.path.join(root, file)) / 1e6
        print(f'{subindent}{file} ({size:.1f}MB)')
    if len(files) > 10:
        print(f'{subindent}... +{len(files)-10} more files')

In [ ]:
# 打印文件大小统计
total_size = 0
for root, dirs, files in os.walk(output_dir):
    for f in files:
        total_size += os.path.getsize(os.path.join(root, f))
print(f'output 总大小: {total_size/1e9:.2f} GB')
print(f'\n训练完成！可在 Kaggle 右侧面板下载 output。')